# SageMaker AI — Comprehensive End-to-End Demo

**Course:** AWS DODEVA / MLGAIE
**Environment:** SageMaker AI Studio → JupyterLab → kernel **Python 3 (SageMaker Distribution)** (CPU)
**Estimated runtime:** ~65 min cell execution + ~25 min teaching = **90 min**
**Estimated cost:** a few US dollars at us-east-1 on-demand pricing.

---

## What you'll learn

Every major SageMaker AI building block, end-to-end, in one notebook:

1. **§3 Processing** — `SKLearnProcessor` for feature engineering at scale.
2. **§4 Training** — built-in XGBoost training job + CloudWatch metrics.
3. **§5 Pipelines** — `Pipeline(Processing → Training → RegisterModel)` with caching.
4. **§6 Deployment** — `ModelPackage.deploy(...)` with **DataCapture** for monitoring.
5. **§7 Invocation** — live predictions; verifying captured payloads in S3.
6. **§8 Autoscaling** — target-tracking on `InvocationsPerInstance`.
7. **§9 Model Cards & Dashboard** — governance metadata that lights up the Studio dashboard.
8. **§10 Monitoring** — `DefaultModelMonitor` with baseline + hourly schedule.

> **Quota & cost note:** Processing jobs use **`ml.t3.large`** (default quota: 4),
> training and pipeline-training use **`ml.m5.xlarge`** (gated by the global "4 training
> instances" quota), and the endpoint uses **`ml.m5.large`** (default quota: 4). Total
> active spend is one endpoint at a time. The §12 cleanup section **must** run before
> you log off — leaving the endpoint running is the expensive bit.

> **Dataset:** California Housing (sklearn built-in). Regression target = median house value.

## §0. Diagnostic + resilient setup

We deliberately do NOT rely on `sagemaker.session.get_execution_role()` here — different
SageMaker Distribution images ship slightly different `sagemaker` layouts. boto3 + STS
gives us the role ARN unambiguously, then the SageMaker SDK is used only for the helpers
we actually need.

In [ ]:
import sys, os, importlib

print("Python:", sys.version.split()[0])

import boto3
print("boto3 :", boto3.__version__)

# 1. Probe the SageMaker SDK so we can see what's installed.
try:
    import sagemaker
    sm_version  = getattr(sagemaker, "__version__", "<no __version__>")
    sm_location = getattr(sagemaker, "__file__", "<no __file__>")
    print(f"sagemaker: {sm_version} @ {sm_location}")
except ImportError as e:
    print("sagemaker SDK not importable:", e)
    print("→ Run §1 install cell, RESTART KERNEL, then re-run §0.")
    raise

# 2. Region
region = boto3.session.Session().region_name or os.environ.get("AWS_REGION") or "us-east-1"
print(f"Region: {region}")

# 3. Account + role ARN via STS
sts = boto3.client("sts", region_name=region)
caller = sts.get_caller_identity()
account = caller["Account"]
arn     = caller["Arn"]

if os.environ.get("SAGEMAKER_ROLE"):
    # Local-execution override: pass an explicit SageMaker execution role ARN.
    role = os.environ["SAGEMAKER_ROLE"]
elif ":assumed-role/" in arn:
    # Inside Studio: the assumed-role ARN points to the execution role.
    role_name = arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
    role      = f"arn:aws:iam::{account}:role/{role_name}"
else:
    role = arn

print(f"Account: {account}")
print(f"Role   : {role}")

# 4. Default bucket
try:
    session = sagemaker.Session()
    bucket  = session.default_bucket()
except Exception as e:
    print(f"sagemaker.Session().default_bucket() failed: {e}")
    session = None
    bucket  = f"sagemaker-{region}-{account}"

print(f"Bucket : {bucket}")
print("✓ setup OK")

## §1. Install / upgrade dependencies

SageMaker Distribution ships with `sagemaker`, `boto3`, `xgboost`, `scikit-learn`, but
we pin a recent SDK because the `sagemaker.workflow` and `sagemaker.model_card` APIs
evolve quickly.

In [ ]:
# Pin to sagemaker 2.x — version 3.x is a separate package with an incompatible API.
%pip install --quiet --upgrade "sagemaker>=2.246.0,<3.0" "boto3>=1.34.0"
print("✓ deps ready — if the SDK was upgraded, restart the kernel before continuing")

## §2. Download the California Housing dataset → S3

sklearn has the dataset bundled — no S3 fetches, no Kaggle keys. We split locally
(75/15/10 train/val/test) and upload three CSVs to S3 so subsequent Processing /
Training / Pipeline cells can all consume the same canonical inputs.

In [ ]:
import datetime as dt, os, io
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

RUN_ID = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
PREFIX = "comprehensive-demo"
print(f"RUN_ID = {RUN_ID}")

cal = fetch_california_housing(as_frame=True)
df  = cal.frame.copy()
# XGBoost CSV convention: target in column 0, no header
target_col = "MedHouseVal"
feature_cols = [c for c in df.columns if c != target_col]
df = df[[target_col] + feature_cols]

train_df, temp_df = train_test_split(df, test_size=0.25, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.40, random_state=42)
print(f"Rows train/val/test = {len(train_df)} / {len(val_df)} / {len(test_df)}")

work = "data"
os.makedirs(work, exist_ok=True)
raw_train = f"{work}/train.csv"; train_df.to_csv(raw_train, index=False, header=False)
raw_val   = f"{work}/validation.csv"; val_df.to_csv(raw_val, index=False, header=False)
raw_test  = f"{work}/test.csv"; test_df.to_csv(raw_test, index=False, header=False)

s3 = boto3.client("s3", region_name=region)
raw_prefix = f"{PREFIX}/{RUN_ID}/raw"
for local in (raw_train, raw_val, raw_test):
    key = f"{raw_prefix}/{os.path.basename(local)}"
    s3.upload_file(local, bucket, key)

raw_uri       = f"s3://{bucket}/{raw_prefix}"      # prefix with all 3 CSVs
raw_train_uri = f"{raw_uri}/train.csv"
raw_val_uri   = f"{raw_uri}/validation.csv"
raw_test_uri  = f"{raw_uri}/test.csv"
print("Uploaded raw CSVs:")
for u in (raw_train_uri, raw_val_uri, raw_test_uri):
    print(" ", u)

## §3. Data Processing — `SKLearnProcessor`

A **Processing Job** runs containerized data-prep code on managed compute, then writes the
outputs back to S3. This is the production-grade equivalent of running pandas in a
notebook. The same script is reusable verbatim in the Pipeline (§5).

What this job does:
1. Reads the raw CSV.
2. Fits a `StandardScaler` on the training partition only (no leakage).
3. Writes `train.csv`, `validation.csv`, `test.csv` to three separate S3 channels —
   the shape XGBoost training expects.

> **Quota:** `ml.t3.large × 1` for ~3 minutes — t3.large for processing is the
> only family with non-zero default quota in fresh AWS accounts (m5.large for
> processing is 0 by default).

In [ ]:
%%writefile preprocessing.py
import os, argparse, pandas as pd
from sklearn.preprocessing import StandardScaler

parser = argparse.ArgumentParser()
parser.add_argument("--input-dir",  default="/opt/ml/processing/input")
parser.add_argument("--train-dir",  default="/opt/ml/processing/output/train")
parser.add_argument("--val-dir",    default="/opt/ml/processing/output/validation")
parser.add_argument("--test-dir",   default="/opt/ml/processing/output/test")
args = parser.parse_args()

for d in (args.train_dir, args.val_dir, args.test_dir):
    os.makedirs(d, exist_ok=True)

# The Processing job mounts each input channel as a separate file under input-dir.
train = pd.read_csv(os.path.join(args.input_dir, "train.csv"),       header=None)
val   = pd.read_csv(os.path.join(args.input_dir, "validation.csv"),  header=None)
test  = pd.read_csv(os.path.join(args.input_dir, "test.csv"),        header=None)

# Column 0 is target, rest are features
y_train, X_train = train.iloc[:, 0], train.iloc[:, 1:]
y_val,   X_val   = val.iloc[:, 0],   val.iloc[:, 1:]
y_test,  X_test  = test.iloc[:, 0],  test.iloc[:, 1:]

scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_val_s   = pd.DataFrame(scaler.transform(X_val),   columns=X_val.columns)
X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

pd.concat([y_train.reset_index(drop=True), X_train_s], axis=1).to_csv(
    os.path.join(args.train_dir, "train.csv"), header=False, index=False)
pd.concat([y_val.reset_index(drop=True),   X_val_s],   axis=1).to_csv(
    os.path.join(args.val_dir,   "validation.csv"), header=False, index=False)
pd.concat([y_test.reset_index(drop=True),  X_test_s],  axis=1).to_csv(
    os.path.join(args.test_dir,  "test.csv"), header=False, index=False)

print("Processing complete.")

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.t3.large",   # default account quota: 4 t3.large for processing
    instance_count=1,
    base_job_name="cal-housing-preprocess",
    sagemaker_session=session,
)

proc_output_prefix = f"{PREFIX}/{RUN_ID}/processed"

processor.run(
    code="preprocessing.py",
    inputs=[
        # ONE input for the whole raw/ prefix — all 3 CSVs land in /opt/ml/processing/input/
        ProcessingInput(source=raw_uri, destination="/opt/ml/processing/input"),
    ],
    outputs=[
        ProcessingOutput(output_name="train",      source="/opt/ml/processing/output/train",
                         destination=f"s3://{bucket}/{proc_output_prefix}/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation",
                         destination=f"s3://{bucket}/{proc_output_prefix}/validation"),
        ProcessingOutput(output_name="test",       source="/opt/ml/processing/output/test",
                         destination=f"s3://{bucket}/{proc_output_prefix}/test"),
    ],
    wait=True,
    logs="None",   # keep notebook output clean; full logs are in CloudWatch
)

train_uri      = f"s3://{bucket}/{proc_output_prefix}/train"
validation_uri = f"s3://{bucket}/{proc_output_prefix}/validation"
test_uri       = f"s3://{bucket}/{proc_output_prefix}/test"
print("Processed S3 channels:")
print(" ", train_uri); print(" ", validation_uri); print(" ", test_uri)

## §4. Training — built-in XGBoost

The Estimator pattern: AWS-maintained container, your hyperparameters, your S3 channels.
Metrics are streamed to CloudWatch automatically; the SDK exposes them via
`TrainingJobAnalytics` so we can populate the Model Card later.

Why XGBoost: it's the workhorse for tabular regression, and the container ships with the
SageMaker training shim out of the box.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker import image_uris

xgb_image_uri = image_uris.retrieve("xgboost", region, version="1.7-1")
print("Training container:", xgb_image_uri)

output_path = f"s3://{bucket}/{PREFIX}/{RUN_ID}/training-output"

xgb = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",   # training quota gates on the global "4 instances" limit
    output_path=output_path,
    sagemaker_session=session,
    base_job_name="cal-housing-xgb",
    max_run=900,
    hyperparameters={
        "objective":   "reg:squarederror",
        "num_round":   100,
        "max_depth":   5,
        "eta":         0.2,
        "subsample":   0.8,
        "eval_metric": "rmse",
    },
)

xgb.fit(
    inputs={
        "train":      TrainingInput(train_uri,      content_type="text/csv"),
        "validation": TrainingInput(validation_uri, content_type="text/csv"),
    },
    wait=True,
    logs="None",
)

training_job_name = xgb.latest_training_job.name
model_artifact    = xgb.model_data
print("Training job:", training_job_name)
print("Model artifact:", model_artifact)

In [ ]:
# Pull the training metrics — used in §9 to populate the Model Card automatically
from sagemaker.analytics import TrainingJobAnalytics
metrics_df = TrainingJobAnalytics(training_job_name=training_job_name).dataframe()
metrics_df

## §5. SageMaker Pipelines — orchestrate the whole thing

Pipelines turn the steps above into a versioned, reproducible DAG that you can:
- kick off with one API call,
- parameterize for different runs,
- **cache** so repeat executions skip unchanged steps,
- visualize as a graph in Studio.

We build a 3-step pipeline: **Preprocess → Train → Register**. With caching enabled and
the same inputs we used in §3+§4, the pipeline's first execution finishes in ~10 min and
a subsequent re-run is ~2 min (cache hits).

> **Best practice:** in a real MLOps setup, this pipeline is the *only* code path that
> produces production models. Notebook training jobs (§4) are for exploration; the
> pipeline is for shipping.

In [ ]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.workflow.step_collections import RegisterModel

pipeline_session = PipelineSession()
cache = CacheConfig(enable_caching=True, expire_after="P30D")

# Pipeline-bound versions of the processor & estimator (use PipelineSession)
pipe_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.t3.large",
    instance_count=1,
    base_job_name="pipe-preprocess",
    sagemaker_session=pipeline_session,
)

raw_input_param = ParameterString(name="RawInputPrefix",
                                  default_value=f"s3://{bucket}/{raw_prefix}")
proc_output_param = ParameterString(name="ProcOutputPrefix",
                                    default_value=f"s3://{bucket}/{PREFIX}/{RUN_ID}/pipeline-processed")

step_preprocess = ProcessingStep(
    name="PreprocessCaliforniaHousing",
    processor=pipe_processor,
    inputs=[
        ProcessingInput(source=raw_uri, destination="/opt/ml/processing/input"),
    ],
    outputs=[
        ProcessingOutput(output_name="train",      source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation"),
        ProcessingOutput(output_name="test",       source="/opt/ml/processing/output/test"),
    ],
    code="preprocessing.py",
    cache_config=cache,
)

pipe_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{PREFIX}/{RUN_ID}/pipeline-output",
    sagemaker_session=pipeline_session,
    base_job_name="pipe-train",
    max_run=900,
    hyperparameters={
        "objective":   "reg:squarederror",
        "num_round":   100,
        "max_depth":   5,
        "eta":         0.2,
        "subsample":   0.8,
        "eval_metric": "rmse",
    },
)

step_train = TrainingStep(
    name="TrainXGBoost",
    estimator=pipe_estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_preprocess.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"),
        "validation": TrainingInput(
            s3_data=step_preprocess.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv"),
    },
    cache_config=cache,
)

MODEL_PACKAGE_GROUP = f"cal-housing-xgb-{RUN_ID}"

step_register = RegisterModel(
    name="RegisterModelPackage",
    estimator=pipe_estimator,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large", "ml.m5.xlarge", "ml.c5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=MODEL_PACKAGE_GROUP,
    approval_status="PendingManualApproval",
)

pipeline_name = f"cal-housing-comprehensive-{RUN_ID}"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[raw_input_param, proc_output_param],
    steps=[step_preprocess, step_train, step_register],
    sagemaker_session=pipeline_session,
)
pipeline.upsert(role_arn=role)
print("Created pipeline:", pipeline_name)

In [ ]:
# Start an execution and wait for it
execution = pipeline.start()
print("Pipeline execution ARN:", execution.arn)
execution.wait(delay=30, max_attempts=120)   # up to 60 min
for step in execution.list_steps():
    print(f"  {step['StepName']:30s}  {step['StepStatus']:10s}  ({step.get('CacheHitResult', {}).get('SourcePipelineExecutionArn', '–')[:60]})")

In [ ]:
# Grab the latest model package version that was registered by the pipeline
sm_client = boto3.client("sagemaker", region_name=region)
pkgs = sm_client.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime", SortOrder="Descending", MaxResults=5,
)["ModelPackageSummaryList"]
latest_pkg_arn = pkgs[0]["ModelPackageArn"]
print("Latest registered model package:", latest_pkg_arn)

# Approve it so we can deploy
sm_client.update_model_package(
    ModelPackageArn=latest_pkg_arn,
    ModelApprovalStatus="Approved",
)
print("Status: Approved")

## §6. Deploy the model — real-time endpoint with **DataCapture**

`ModelPackage.deploy(...)` provisions:
- a **Model** (immutable container + model artifact),
- an **Endpoint Config** (instance type + count + capture settings),
- an **Endpoint** (the live inference URL).

We turn on **DataCapture** at this layer — that's what feeds Model Monitor (§10).
Without `data_capture_config`, the monitor has nothing to compare against.

In [ ]:
from sagemaker.model import ModelPackage
from sagemaker.model_monitor import DataCaptureConfig

capture_prefix = f"s3://{bucket}/{PREFIX}/{RUN_ID}/data-capture"

capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,        # demo: capture everything; 10–20% is normal in prod
    destination_s3_uri=capture_prefix,
    capture_options=["Input", "Output"],
    sagemaker_session=session,
)

endpoint_name = f"cal-housing-ep-{RUN_ID}"

model_pkg = ModelPackage(
    role=role,
    model_package_arn=latest_pkg_arn,
    sagemaker_session=session,
)

model_pkg.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    data_capture_config=capture_config,
    wait=True,
)

# ModelPackage.deploy() returns None when no predictor_cls is set on the package,
# so we construct a generic Predictor ourselves (the endpoint is already InService).
from sagemaker.predictor import Predictor
predictor = Predictor(endpoint_name=endpoint_name, sagemaker_session=session)
print("Endpoint live:", predictor.endpoint_name)

## §7. Live invocation + DataCapture verification

In [ ]:
from sagemaker.serializers   import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

predictor.serializer   = CSVSerializer()
predictor.deserializer = CSVDeserializer()

# Use the held-out test set — read from the post-processing test channel
s3_obj = s3.get_object(Bucket=bucket, Key=f"{proc_output_prefix}/test/test.csv")
test_df = pd.read_csv(io.BytesIO(s3_obj["Body"].read()), header=None)
y_true  = test_df.iloc[:10, 0].values
X_eval  = test_df.iloc[:10, 1:].values

preds_raw = predictor.predict(X_eval)
# CSVDeserializer returns a list-of-lists of strings
y_pred = np.array([float(r[0]) for r in preds_raw])

print("First 10 test rows:")
print(pd.DataFrame({"y_true": y_true, "y_pred": y_pred.round(3),
                    "abs_err": (y_true - y_pred).round(3)}).to_string(index=False))

In [ ]:
# Prove DataCapture captured the request — list S3 prefix, give it ~30s to settle
import time
time.sleep(30)
capture_key_prefix = f"{PREFIX}/{RUN_ID}/data-capture/{endpoint_name}"
resp = s3.list_objects_v2(Bucket=bucket, Prefix=capture_key_prefix)
keys = [obj["Key"] for obj in resp.get("Contents", [])]
print(f"Capture files under s3://{bucket}/{capture_key_prefix}:")
for k in keys[-5:]:
    print(" ", k)
if not keys:
    print("  (no captures yet — they appear within ~1 min after invocation)")

## §8. Auto-scaling the endpoint

Target-tracking policy on the built-in metric `SageMakerVariantInvocationsPerInstance`.
SageMaker scales out when the average per-instance invocation count exceeds the target,
and scales in when it stays below.

> In production: **always** load-test before trusting your scaling policy. The
> 60s scale-out / 300s scale-in cooldowns matter when traffic is bursty.

In [ ]:
aas = boto3.client("application-autoscaling", region_name=region)
resource_id = f"endpoint/{endpoint_name}/variant/AllTraffic"

aas.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=1,
    MaxCapacity=4,
)

aas.put_scaling_policy(
    PolicyName=f"target-invocations-{RUN_ID}",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 20.0,
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"
        },
        "ScaleOutCooldown": 60,
        "ScaleInCooldown":  300,
    },
)
print(f"Autoscaling registered on {resource_id} (1→4 instances, target=20 req/inst/min)")

## §9. Model Card — governance & the Studio Model Dashboard

Model Cards are SageMaker's first-class governance artifact. A populated card shows up
in the **Studio Model Dashboard**, gives reviewers context (intended use, training data,
metrics, ethical considerations), and can be exported to PDF for compliance reviews.

We auto-populate the metric values from the §4 `TrainingJobAnalytics` so the numbers
stay in sync with the actual training run.

In [ ]:
from sagemaker.model_card import (
    ModelCard, ModelOverview, TrainingDetails, EvaluationJob, MetricGroup,
    Metric, MetricTypeEnum, ModelCardStatusEnum, IntendedUses,
    BusinessDetails, AdditionalInformation, TrainingJobDetails,
    TrainingMetric, HyperParameter, ObjectiveFunction,
)
from sagemaker.model_card.model_card import Function
from sagemaker.model_card.schema_constraints import ObjectiveFunctionEnum, FacetEnum

# Pull the final RMSE / MAE from CloudWatch metrics (validation channel)
val_rmse = float(metrics_df[metrics_df["metric_name"] == "validation:rmse"]["value"].iloc[-1])                   if not metrics_df[metrics_df["metric_name"] == "validation:rmse"].empty else float("nan")

model_overview = ModelOverview(
    model_description="XGBoost regressor on California Housing — predicts median house value.",
    model_owner="aws-classroom-demo",
    model_creator="comprehensive-demo-notebook",
    problem_type="Regression",
    algorithm_type="XGBoost",
    model_artifact=[model_artifact],
)

intended_uses = IntendedUses(
    purpose_of_model="Pedagogy: showcase end-to-end SageMaker workflow.",
    intended_uses="Demonstration only. NOT for housing-price decisions in the real world.",
    factors_affecting_model_efficiency="Older 1990 census data; California-only; 8 features.",
    risk_rating="Low",   # SDK validates against {'High', 'Low', 'Medium', 'Unknown'}
    explanations_for_risk_rating="No production deployment, no PII, no decisions.",
)

training_details = TrainingDetails(
    objective_function=ObjectiveFunction(
        function=Function(
            function=ObjectiveFunctionEnum.MINIMIZE,
            facet=FacetEnum.RMSE,
            condition="Lower is better.",
        ),
    ),
    training_observations="Trained on 75% of the California Housing dataset; "
                          "StandardScaler applied to features only.",
    training_job_details=TrainingJobDetails(
        training_arn=xgb.latest_training_job.describe()["TrainingJobArn"],
        training_datasets=[raw_train_uri],
        training_metrics=[TrainingMetric(name="validation:rmse", value=val_rmse)],
        user_provided_training_metrics=[],
        hyper_parameters=[
            HyperParameter(name="num_round",   value="100"),
            HyperParameter(name="max_depth",   value="5"),
            HyperParameter(name="eta",         value="0.2"),
            HyperParameter(name="objective",   value="reg:squarederror"),
        ],
    ),
)

eval_job = EvaluationJob(
    name="held-out-test-rmse",
    datasets=[raw_test_uri],
    metric_groups=[
        MetricGroup(
            name="regression-metrics",
            metric_data=[Metric(name="rmse",
                                type=MetricTypeEnum.NUMBER,
                                value=val_rmse)],
        )
    ],
)

additional_info = AdditionalInformation(
    ethical_considerations="California housing data is from 1990 and reflects historical "
                           "patterns; model should not be used for any real housing or "
                           "lending decisions. Demonstration purposes only.",
    caveats_and_recommendations="Re-train on contemporary data and re-evaluate fairness "
                                "across protected attributes before any real-world use.",
    custom_details={"course": "AWS DODEVA / MLGAIE", "demo_run_id": RUN_ID},
)

business = BusinessDetails(
    business_problem="Predict median home value (regression) — pedagogical demo.",
    business_stakeholders="AWS classroom learners",
    line_of_business="Education / Training",
)

model_card_name = f"cal-housing-card-{RUN_ID}"
card = ModelCard(
    name=model_card_name,
    status=ModelCardStatusEnum.DRAFT,
    model_overview=model_overview,
    intended_uses=intended_uses,
    training_details=training_details,
    evaluation_details=[eval_job],
    additional_information=additional_info,
    business_details=business,
    sagemaker_session=session,
)
card.create()
print(f"Created Model Card: {model_card_name}")
print("→ Open it in Studio: Models → Model Dashboard → Model Cards")

## §10. Model Monitor — baseline + hourly schedule

Model Monitor compares **live captured traffic** (§6 DataCapture) against a **baseline**
of statistics + constraints derived from training data. We:

1. Run `suggest_baseline()` on the post-processing training data → produces
   `statistics.json` and `constraints.json` in S3.
2. Create a `MonitoringSchedule` that runs hourly. The first report fires at the top
   of the next hour; meaningful drift detection wants ≥24 h of capture.

> For class: we set the loop up here so you can see the wiring, then poke at the
> captured-data S3 prefix (it's been growing since §7) to convince yourself the
> pipeline is real.

In [ ]:
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.model_monitor import CronExpressionGenerator
from sagemaker.model_monitor import EndpointInput

monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.t3.large",   # monitor uses processing-job quotas
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)

baseline_uri = f"s3://{bucket}/{PREFIX}/{RUN_ID}/baseline"
monitor.suggest_baseline(
    baseline_dataset=f"{train_uri}/train.csv",
    dataset_format=DatasetFormat.csv(header=False),
    output_s3_uri=baseline_uri,
    wait=True,
    logs=False,
)
print("Baseline files at:", baseline_uri)

In [ ]:
schedule_name = f"cal-housing-monitor-{RUN_ID}"
monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name,
    endpoint_input=EndpointInput(
        endpoint_name=endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
    ),
    output_s3_uri=f"s3://{bucket}/{PREFIX}/{RUN_ID}/monitor-reports",
    statistics=monitor.baseline_statistics(),
    constraints=monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

desc = sm_client.describe_monitoring_schedule(MonitoringScheduleName=schedule_name)
print(f"Monitoring schedule: {schedule_name}")
print(f"  status: {desc['MonitoringScheduleStatus']}")
print(f"  cron : {desc['MonitoringScheduleConfig']['ScheduleConfig']['ScheduleExpression']}")
print(f"→ First execution at the top of the next hour. Drift signals need 24h+ of capture.")

## §11. Class discussion prompts

1. **Pipelines vs. notebooks:** what changes when you have to re-train weekly with the
   same code path? Why is the pipeline cache load-bearing?
2. **DataCapture sampling:** we set `sampling_percentage=100` for the demo. What's the
   cost trade-off in production? (Hint: every captured record is an extra S3 PUT and
   extra storage.)
3. **Model Card status:** we left ours in `Draft`. Who should have permission to
   transition to `PendingPublish` / `Published`, and what gates would you put in front
   of that transition?
4. **Auto-scaling target value:** we used `InvocationsPerInstance=20`. How would you
   pick this number in real life? (Hint: load-test, find the inflection point where
   latency degrades, then set target to ~70% of that.)
5. **Monitor → action:** what should *happen* when the schedule reports a constraint
   violation? Page the on-call? Block deployments? Open a Jira ticket?

## §12. Cleanup — **don't skip this**

The endpoint is the costly bit (~$0.10/h). The monitoring schedule re-launches a
Processing job every hour (~$0.20/h). Everything else is pennies. Run all four cells
below before you log off.

In [ ]:
# 12.1 — stop & delete the monitoring schedule
try:
    sm_client.stop_monitoring_schedule(MonitoringScheduleName=schedule_name)
except Exception as e:
    print("stop schedule:", e)
try:
    sm_client.delete_monitoring_schedule(MonitoringScheduleName=schedule_name)
    print(f"deleted monitoring schedule {schedule_name}")
except Exception as e:
    print("delete schedule:", e)

In [ ]:
# 12.2 — deregister autoscaling, delete endpoint + endpoint config + model
try:
    aas.deregister_scalable_target(
        ServiceNamespace="sagemaker",
        ResourceId=resource_id,
        ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    )
    print("deregistered autoscaling target")
except Exception as e:
    print("deregister autoscaling:", e)

try:
    predictor.delete_endpoint(delete_endpoint_config=True)
    print("deleted endpoint + endpoint-config")
except Exception as e:
    print("delete endpoint:", e)

try:
    predictor.delete_model()
    print("deleted model")
except Exception as e:
    print("delete model:", e)

In [ ]:
# 12.3 — model package + group + model card + pipeline
try:
    sm_client.delete_model_package(ModelPackageName=latest_pkg_arn)
    print("deleted model package")
except Exception as e:
    print("delete model package:", e)

try:
    sm_client.delete_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
    print("deleted model package group")
except Exception as e:
    print("delete model package group:", e)

try:
    sm_client.delete_model_card(ModelCardName=model_card_name)
    print("deleted model card")
except Exception as e:
    print("delete model card:", e)

try:
    sm_client.delete_pipeline(PipelineName=pipeline_name)
    print("deleted pipeline")
except Exception as e:
    print("delete pipeline:", e)

In [ ]:
# 12.4 — sweep S3 demo prefix
s3_resource = boto3.resource("s3", region_name=region)
bucket_obj  = s3_resource.Bucket(bucket)
deleted = 0
for obj in bucket_obj.objects.filter(Prefix=f"{PREFIX}/{RUN_ID}/"):
    obj.delete(); deleted += 1
print(f"deleted {deleted} S3 objects under {PREFIX}/{RUN_ID}/")

---

### Recap

In ~90 minutes you exercised every major SageMaker AI surface:

- **Processing** for repeatable feature engineering.
- **Training** with a built-in algorithm and CloudWatch-backed metrics.
- **Pipelines** that wrap Processing + Training + Register into a single DAG with caching.
- **Model Registry + Approval** as the gate between "trained" and "deployed".
- **Real-time endpoints** with **DataCapture** wired in from day one.
- **Auto-scaling** for elastic capacity.
- **Model Cards** to make governance a first-class artifact.
- **Model Monitor** to close the feedback loop between live traffic and the trained-on distribution.

Every resource is now torn down — verify with `aws sagemaker list-endpoints` if you want to be sure.